# 04 · 敏感性分析

本 Notebook 用于：
1. PRCC 偏秩相关敏感性分析（SALib LHS 采样）
2. 龙卷风图可视化（3个目标量：峰值感染率、累计发病率、峰值时间）
3. R₀ 参数热力图（β₀ × γ）
4. 单参数扫描折线图

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from model.params import ModelParams
from model.r0 import compute_R0, r0_heatmap_data, r0_sensitivity_table
from sensitivity.prcc import run_prcc_analysis
from plots._style import setup_style
from plots.sensitivity_plot import plot_prcc_tornado, plot_r0_heatmap, plot_r0_sensitivity_line

setup_style()

FIG_DIR  = Path('../output/figures')
SENS_DIR = Path('../output/sensitivity')
FIG_DIR.mkdir(parents=True, exist_ok=True)
SENS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
p = ModelParams.from_yaml('../config.yaml') if Path('../config.yaml').exists() else ModelParams()
R0 = compute_R0(p)
print(f'基础参数 R₀ = {R0:.4f}')

## 1. PRCC 敏感性分析

In [ ]:
# n_samples=500 约需 3-5 分钟；快速测试用 200
prcc_result = run_prcc_analysis(
    p,
    n_samples=500,
    param_variation=0.30,
    seed=42,
    output_dir=SENS_DIR,
)

print(f"有效样本数: {prcc_result['n_valid']}")

In [ ]:
# 显示各目标量的 PRCC 表
for out_name, df_prcc in prcc_result['prcc'].items():
    print(f'\n=== {out_name} ===')
    print(df_prcc[['parameter','label','PRCC','p_value']].to_string(index=False))

In [ ]:
# 绘制 Tornado 图（论文必图）
for out_name, df_prcc in prcc_result['prcc'].items():
    fig = plot_prcc_tornado(
        df_prcc,
        output_name=out_name,
        title='PRCC 敏感性分析（龙卷风图）',
        output_path=FIG_DIR / f'04_prcc_{out_name}.png',
        show=True
    )

## 2. R₀ 参数热力图

In [ ]:
x_vals = np.linspace(0.10, 0.60, 40)  # β₀
y_vals = np.linspace(0.10, 0.50, 35)  # γ
Z = r0_heatmap_data(p, 'beta0', 'gamma', x_vals, y_vals)

fig = plot_r0_heatmap(
    Z, x_vals, y_vals,
    param_x='beta0', param_y='gamma',
    title=f'R₀ 参数热力图（α={p.alpha:.2f}）',
    output_path=FIG_DIR / '04_r0_heatmap.png',
    show=True
)

## 3. 单参数扫描

In [ ]:
params_to_scan = {
    'beta0':  (np.linspace(0.05, 0.70, 50), '基础传播系数 β₀'),
    'alpha':  (np.linspace(0.01, 0.80, 50), '病例隔离率 α'),
    'delta1': (np.linspace(0.00, 0.70, 50), '年季节振幅 δ₁'),
}

for param, (values, label) in params_to_scan.items():
    tbl = r0_sensitivity_table(p, param, values)
    base_v = getattr(p, param)
    fig = plot_r0_sensitivity_line(
        np.array(tbl['values']), tbl['R0'], tbl['HIT'],
        param_label=label,
        base_val=base_v,
        output_path=FIG_DIR / f'04_r0_{param}_sensitivity.png',
        show=True
    )
    print(f'{label}: 当前值={base_v:.3f} → R₀={compute_R0(p):.3f}')